In [1]:
import os
import shutil
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn

from PIL import Image
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models

In [2]:
print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))


PyTorch version: 2.11.0+cu128
CUDA available: True
GPU: Tesla T4


In [3]:
from google.colab import drive

drive.mount("/content/drive")


Mounted at /content/drive


In [4]:
DRIVE_DATA = Path("/content/drive/MyDrive/bone_tumor_data/final")
LOCAL_DATA = Path("/content/bone_tumor_data/final")

if LOCAL_DATA.exists():
    shutil.rmtree(LOCAL_DATA)

shutil.copytree(DRIVE_DATA, LOCAL_DATA)

print("Dataset copied successfully.")
print("Local dataset path:", LOCAL_DATA)

Dataset copied successfully.
Local dataset path: /content/bone_tumor_data/final


In [5]:

DATA_ROOT = Path("/content/bone_tumor_data/final")

print("Dataset root:", DATA_ROOT)

Dataset root: /content/bone_tumor_data/final


In [6]:
for split in ["train", "valid", "test"]:
    image_dir = DATA_ROOT / split / "images"
    metadata_file = DATA_ROOT / split / "metadata.csv"

    image_count = len(list(image_dir.glob("*.png")))
    metadata_count = len(pd.read_csv(metadata_file))

    print(
        f"{split}: "
        f"{image_count} images, "
        f"{metadata_count} metadata rows"
    )

train: 10052 images, 10052 metadata rows
valid: 1084 images, 1084 metadata rows
test: 1067 images, 1067 metadata rows


In [7]:
for split in ["train", "valid", "test"]:
    metadata = pd.read_csv(
        DATA_ROOT / split / "metadata.csv"
    )

    print(f"\n{split} label distribution:")
    print(metadata["cancer"].value_counts().sort_index())


train label distribution:
cancer
0    6697
1    3355
Name: count, dtype: int64

valid label distribution:
cancer
0    653
1    431
Name: count, dtype: int64

test label distribution:
cancer
0    650
1    417
Name: count, dtype: int64


In [8]:
train_transform = transforms.Compose([
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

valid_test_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

In [9]:
class BoneTumorDataset(Dataset):

    def __init__(self, split, transform=None):
        self.split = split
        self.transform = transform

        self.image_dir = DATA_ROOT / split / "images"

        metadata = pd.read_csv(
            DATA_ROOT / split / "metadata.csv"
        )

        self.metadata = metadata[
            metadata["filename"].apply(
                lambda x: (self.image_dir / x).exists()
            )
        ].reset_index(drop=True)

    def __len__(self):
        return len(self.metadata)

    def __getitem__(self, index):

        row = self.metadata.iloc[index]

        image_path = self.image_dir / row["filename"]

        image = Image.open(
            image_path
        ).convert("RGB")

        label = int(row["cancer"])

        if self.transform:
            image = self.transform(image)

        return image, label

In [10]:
train_dataset = BoneTumorDataset(
    split="train",
    transform=train_transform
)

valid_dataset = BoneTumorDataset(
    split="valid",
    transform=valid_test_transform
)

test_dataset = BoneTumorDataset(
    split="test",
    transform=valid_test_transform
)

print("Train dataset:", len(train_dataset))
print("Validation dataset:", len(valid_dataset))
print("Test dataset:", len(test_dataset))

Train dataset: 10052
Validation dataset: 1084
Test dataset: 1067


In [11]:
BATCH_SIZE = 32

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=0
)

valid_loader = DataLoader(
    valid_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0
)

print("DataLoaders created successfully.")

DataLoaders created successfully.


In [12]:
images, labels = next(iter(train_loader))

print("Image batch shape:", images.shape)
print("Label batch shape:", labels.shape)
print("Unique labels:", torch.unique(labels))

Image batch shape: torch.Size([32, 3, 224, 224])
Label batch shape: torch.Size([32])
Unique labels: tensor([0, 1])


In [13]:
model = models.efficientnet_b0(
    weights=models.EfficientNet_B0_Weights.DEFAULT
)

print("EfficientNet-B0 loaded with ImageNet pretrained weights.")

Downloading: "https://download.pytorch.org/models/efficientnet_b0_rwightman-7f5810bc.pth" to /root/.cache/torch/hub/checkpoints/efficientnet_b0_rwightman-7f5810bc.pth


100%|██████████| 20.5M/20.5M [00:00<00:00, 72.8MB/s]


EfficientNet-B0 loaded with ImageNet pretrained weights.


In [14]:
model.classifier[1] = nn.Linear(
    in_features=model.classifier[1].in_features,
    out_features=2
)

print(model.classifier)

Sequential(
  (0): Dropout(p=0.2, inplace=True)
  (1): Linear(in_features=1280, out_features=2, bias=True)
)


In [15]:
for param in model.parameters():
    param.requires_grad = False

for param in model.classifier[1].parameters():
    param.requires_grad = True

total_params = sum(
    p.numel()
    for p in model.parameters()
)

trainable_params = sum(
    p.numel()
    for p in model.parameters()
    if p.requires_grad
)

print("Total parameters:", total_params)
print("Trainable parameters:", trainable_params)

Total parameters: 4010110
Trainable parameters: 2562


In [16]:
device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

model = model.to(device)

print("Using device:", device)

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

Using device: cuda
GPU: Tesla T4


In [17]:
criterion = nn.CrossEntropyLoss()

optimizer = torch.optim.Adam(
    model.classifier[1].parameters(),
    lr=0.001
)

print("Loss function and optimizer ready.")

Loss function and optimizer ready.


In [18]:
def train_one_epoch(
    model,
    loader,
    criterion,
    optimizer,
    device
):

    model.train()

    running_loss = 0.0
    correct = 0
    total = 0

    for images, labels in loader:

        images = images.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()

        outputs = model(images)

        loss = criterion(
            outputs,
            labels
        )

        loss.backward()

        optimizer.step()

        running_loss += (
            loss.item() * images.size(0)
        )

        predictions = outputs.argmax(
            dim=1
        )

        correct += (
            predictions == labels
        ).sum().item()

        total += labels.size(0)

    epoch_loss = running_loss / total
    epoch_accuracy = correct / total

    return epoch_loss, epoch_accuracy

In [19]:
def validate_one_epoch(
    model,
    loader,
    criterion,
    device
):

    model.eval()

    running_loss = 0.0
    correct = 0
    total = 0

    with torch.no_grad():

        for images, labels in loader:

            images = images.to(device)
            labels = labels.to(device)

            outputs = model(images)

            loss = criterion(
                outputs,
                labels
            )

            running_loss += (
                loss.item() * images.size(0)
            )

            predictions = outputs.argmax(
                dim=1
            )

            correct += (
                predictions == labels
            ).sum().item()

            total += labels.size(0)

    epoch_loss = running_loss / total
    epoch_accuracy = correct / total

    return epoch_loss, epoch_accuracy

In [20]:
best_val_accuracy = 0.0

best_model_path = (
    "/content/drive/MyDrive/"
    "efficientnetb0_imagenet_best.pth"
)

num_epochs = 3

history = {
    "train_loss": [],
    "train_accuracy": [],
    "valid_loss": [],
    "valid_accuracy": []
}

for epoch in range(num_epochs):

    train_loss, train_accuracy = train_one_epoch(
        model,
        train_loader,
        criterion,
        optimizer,
        device
    )

    valid_loss, valid_accuracy = validate_one_epoch(
        model,
        valid_loader,
        criterion,
        device
    )

    history["train_loss"].append(train_loss)
    history["train_accuracy"].append(train_accuracy)
    history["valid_loss"].append(valid_loss)
    history["valid_accuracy"].append(valid_accuracy)

    print(
        f"Epoch {epoch + 1}/{num_epochs}"
    )

    print(
        f"Train Loss: {train_loss:.4f} | "
        f"Train Acc: {train_accuracy:.4f}"
    )

    print(
        f"Valid Loss: {valid_loss:.4f} | "
        f"Valid Acc: {valid_accuracy:.4f}"
    )

    if valid_accuracy > best_val_accuracy:

        best_val_accuracy = valid_accuracy

        torch.save(
            model.state_dict(),
            best_model_path
        )

        print("Best model saved.")

print("\nTraining complete.")
print(
    f"Best validation accuracy: "
    f"{best_val_accuracy:.4f}"
)
print(
    f"Best model path: "
    f"{best_model_path}"
)

Epoch 1/3
Train Loss: 0.4010 | Train Acc: 0.8319
Valid Loss: 0.3397 | Valid Acc: 0.8598
Best model saved.
Epoch 2/3
Train Loss: 0.3380 | Train Acc: 0.8615
Valid Loss: 0.3396 | Valid Acc: 0.8533
Epoch 3/3
Train Loss: 0.3284 | Train Acc: 0.8657
Valid Loss: 0.3203 | Valid Acc: 0.8699
Best model saved.

Training complete.
Best validation accuracy: 0.8699
Best model path: /content/drive/MyDrive/efficientnetb0_imagenet_best.pth


In [21]:
model.load_state_dict(
    torch.load(
        best_model_path,
        map_location=device
    )
)

model = model.to(device)
model.eval()

print("Best EfficientNet-B0 model loaded.")
print(
    f"Best validation accuracy: "
    f"{best_val_accuracy:.4f}"
)

Best EfficientNet-B0 model loaded.
Best validation accuracy: 0.8699


In [22]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix
)

all_labels = []
all_predictions = []
all_probabilities = []

model.eval()

with torch.no_grad():

    for images, labels in test_loader:

        images = images.to(device)

        outputs = model(images)

        probabilities = torch.softmax(
            outputs,
            dim=1
        )

        predictions = outputs.argmax(
            dim=1
        )

        all_labels.extend(
            labels.numpy()
        )

        all_predictions.extend(
            predictions.cpu().numpy()
        )

        all_probabilities.extend(
            probabilities[:, 1]
            .cpu()
            .numpy()
        )

accuracy = accuracy_score(
    all_labels,
    all_predictions
)

precision = precision_score(
    all_labels,
    all_predictions
)

recall = recall_score(
    all_labels,
    all_predictions
)

f1 = f1_score(
    all_labels,
    all_predictions
)

auc = roc_auc_score(
    all_labels,
    all_probabilities
)

cm = confusion_matrix(
    all_labels,
    all_predictions
)

tn, fp, fn, tp = cm.ravel()

specificity = tn / (tn + fp)

print(
    "Final Test Results — "
    "EfficientNet-B0 + ImageNet"
)

print("-" * 60)

print(
    f"Accuracy:    "
    f"{accuracy:.4f} "
    f"({accuracy*100:.2f}%)"
)

print(
    f"Precision:   "
    f"{precision:.4f} "
    f"({precision*100:.2f}%)"
)

print(
    f"Recall:      "
    f"{recall:.4f} "
    f"({recall*100:.2f}%)"
)

print(
    f"Specificity: "
    f"{specificity:.4f} "
    f"({specificity*100:.2f}%)"
)

print(
    f"F1 Score:    "
    f"{f1:.4f} "
    f"({f1*100:.2f}%)"
)

print(
    f"ROC-AUC:     "
    f"{auc:.4f} "
    f"({auc*100:.2f}%)"
)

print("\nConfusion Matrix:")
print(cm)

print("\nTN:", tn)
print("FP:", fp)
print("FN:", fn)
print("TP:", tp)

Final Test Results — EfficientNet-B0 + ImageNet
------------------------------------------------------------
Accuracy:    0.8707 (87.07%)
Precision:   0.8345 (83.45%)
Recall:      0.8345 (83.45%)
Specificity: 0.8938 (89.38%)
F1 Score:    0.8345 (83.45%)
ROC-AUC:     0.9393 (93.93%)

Confusion Matrix:
[[581  69]
 [ 69 348]]

TN: 581
FP: 69
FN: 69
TP: 348


In [23]:
results = pd.DataFrame({
    "Metric": [
        "Accuracy",
        "Precision",
        "Recall/Sensitivity",
        "Specificity",
        "F1 Score",
        "ROC-AUC"
    ],
    "Value": [
        accuracy,
        precision,
        recall,
        specificity,
        f1,
        auc
    ]
})

results

,Metric,Value
0,Accuracy,0.870665
1,Precision,0.834532
2,Recall/Sensitivity,0.834532
3,Specificity,0.893846
4,F1 Score,0.834532
5,ROC-AUC,0.939277
